# Startup Search Agent

1. 사용자 입력
2. LLM 키워드 확장
3. YC + 혁신의숲 검색
4. LLM relevance filtering
5. LLM state 생성

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any, TypedDict

from openai import OpenAI

PROJECT_ROOT = Path.cwd()
AGENT_DIR = PROJECT_ROOT if (PROJECT_ROOT / "startup_search_agent.py").exists() else PROJECT_ROOT / "30-Projects" / "robotics-investment-agent"
if AGENT_DIR.exists() and str(AGENT_DIR) not in sys.path:
    sys.path.append(str(AGENT_DIR))

from startup_search_agent import deduplicate_candidates, fetch_yc_candidates, search_innoforest_candidates


def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file(PROJECT_ROOT / ".env")
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("OPNEAI_API_KEY")
if not api_key:
    raise RuntimeError(".env에 OPENAI_API_KEY를 넣어야 합니다. 이전 오타 키 OPNEAI_API_KEY도 임시 지원합니다.")

client = OpenAI(api_key=api_key)
print("client_ready")

In [21]:
class InvestmentState(TypedDict):
    startup_name: str
    startup_list: list[str]
    evaluated_startups: list[str]
    startup_basic_info: dict[str, Any]
    tech_summary: str
    market_evaluation: str
    competitor_analysis: str
    scorecard: dict[str, float]
    final_score: float
    investment_decision: str
    decision_reason: str
    report_content: str
    rag_sources: list[str]

## 1. 사용자 입력

In [33]:
user_query = "요즘 핫한 휴머노이드 관련된 스타트업 정보를 보고 싶어"
user_query

'요즘 핫한 휴머노이드 관련된 스타트업 정보를 보고 싶어'

## 2. LLM 키워드 확장

In [34]:
def extract_search_keywords(user_query: str, client: OpenAI) -> list[str]:
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "system",
                "content": (
                    "You expand user queries into web search keywords for robotics startup discovery. "
                    "Return strict JSON with a single key keywords. "
                    "Generate 4 to 8 focused keywords in Korean and English when useful. "
                    "Prefer product/category terms over generic terms."
                ),
            },
            {
                "role": "user",
                "content": (
                    "사용자 요청: " + user_query + "\n"
                    "예시 형식: {\"keywords\": [\"휴머노이드\", \"humanoid\", \"bipedal robot\", \"general-purpose humanoid\"]}"
                ),
            },
        ],
    )
    payload = json.loads(response.output_text)
    return payload["keywords"]


keywords = extract_search_keywords(user_query, client)
keywords

['휴머노이드 스타트업',
 'humanoid robot startup',
 '휴머노이드 로봇 개발',
 'humanoid robotics companies',
 '인간형 로봇 스타트업',
 'bipedal robot startups',
 'general-purpose humanoid robots']

## 3. YC + 혁신의숲 검색

In [35]:
yc_candidates = fetch_yc_candidates(keywords, hits_per_keyword=8)
innoforest_candidates = search_innoforest_candidates(keywords, max_candidates=20)
raw_candidates = deduplicate_candidates([*yc_candidates, *innoforest_candidates])
len(raw_candidates)

83

In [36]:
for candidate in raw_candidates[:30]:
    print(f"- {candidate.name} [{candidate.source}] | {candidate.sector} | {candidate.stage}")

- Pina Earth [ycombinator] | Industrials -> Climate | Early
- Kaigo Health [ycombinator] | Healthcare -> Healthcare IT | Early
- K-Scale Labs [ycombinator] | Consumer -> Consumer Electronics | Early
- Piggy Robotics [ycombinator] | Industrials -> Manufacturing and Robotics | Early
- Proception Inc [ycombinator] | Industrials -> Manufacturing and Robotics | Early
- Vassar Robotics [ycombinator] | Consumer -> Consumer Electronics | Early
- Human Archive [ycombinator] | Industrials -> Manufacturing and Robotics | Early
- Hlabs [ycombinator] | Industrials | Early
- Cortex AI [ycombinator] | Industrials -> Manufacturing and Robotics | Early
- Yondu [ycombinator] | Industrials -> Manufacturing and Robotics | Early
- RoboTire [ycombinator] | Industrials -> Automotive | Growth
- Double Robotics [ycombinator] | B2B -> Productivity | Early
- Origami Robotics [ycombinator] | Industrials | Early
- One Robot [ycombinator] | Industrials -> Manufacturing and Robotics | Early
- Forge Robotics [ycombin

## 4. LLM Relevance Filtering

In [37]:
def llm_relevance_filter(user_query: str, candidates: list[Any], client: OpenAI) -> list[dict[str, Any]]:
    candidate_payload = [
        {
            "name": candidate.name,
            "source": candidate.source,
            "url": candidate.url,
            "description": candidate.description,
            "location": candidate.location,
            "sector": candidate.sector,
            "stage": candidate.stage,
            "tags": candidate.tags or [],
        }
        for candidate in candidates[:30]
    ]
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "system",
                "content": (
                    "You filter startup search results for an investment analyst. "
                    "Return strict JSON with key filtered_candidates. "
                    "Each item must contain name, relevance_label, relevance_reason. "
                    "Allowed labels are relevant, maybe, irrelevant. "
                    "Judge relevance against the user's query, not against a fixed category list."
                ),
            },
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "user_query": user_query,
                        "candidates": candidate_payload,
                    },
                    ensure_ascii=False,
                ),
            },
        ],
    )
    return json.loads(response.output_text)["filtered_candidates"]


relevance_results = llm_relevance_filter(user_query, raw_candidates, client)
relevance_results

[{'name': 'K-Scale Labs',
  'relevance_label': 'relevant',
  'relevance_reason': 'Focuses on open-source humanoid robots, directly matching the query about hot humanoid-related startups.'},
 {'name': 'Piggy Robotics',
  'relevance_label': 'relevant',
  'relevance_reason': 'Develops humanoid robots for household chores, very relevant to humanoid-related startups.'},
 {'name': 'Proception Inc',
  'relevance_label': 'relevant',
  'relevance_reason': 'Works on making humanoids dexterous enough to thread a needle, clearly related to humanoid robotics.'},
 {'name': 'Vassar Robotics',
  'relevance_label': 'relevant',
  'relevance_reason': 'Offers an open-source semi-humanoid development kit, aligning well with the humanoid startup interest.'},
 {'name': 'Human Archive',
  'relevance_label': 'maybe',
  'relevance_reason': 'Provides multimodal data for robotics but not specifically humanoid-focused; somewhat relevant.'},
 {'name': 'Hlabs',
  'relevance_label': 'maybe',
  'relevance_reason': 'Bu

In [27]:
relevant_names = {
    item["name"]
    for item in relevance_results
    if item["relevance_label"] in {"relevant", "maybe"}
}
filtered_candidates = [candidate for candidate in raw_candidates if candidate.name in relevant_names]
len(filtered_candidates)

12

In [28]:
for candidate in filtered_candidates:
    print(f"- {candidate.name} [{candidate.source}] | {candidate.sector}")

- K-Scale Labs [ycombinator] | Consumer -> Consumer Electronics
- Piggy Robotics [ycombinator] | Industrials -> Manufacturing and Robotics
- Proception Inc [ycombinator] | Industrials -> Manufacturing and Robotics
- Vassar Robotics [ycombinator] | Consumer -> Consumer Electronics
- Human Archive [ycombinator] | Industrials -> Manufacturing and Robotics
- Hlabs [ycombinator] | Industrials
- Cortex AI [ycombinator] | Industrials -> Manufacturing and Robotics
- Origami Robotics [ycombinator] | Industrials
- One Robot [ycombinator] | Industrials -> Manufacturing and Robotics
- Forge Robotics [ycombinator] | Industrials -> Manufacturing and Robotics
- Verne Robotics [ycombinator] | Industrials -> Manufacturing and Robotics
- Boost Robotics [ycombinator] | Industrials -> Manufacturing and Robotics


## 5. LLM State 생성

In [29]:
def build_search_state(user_query: str, candidates: list[Any], client: OpenAI) -> dict[str, Any]:
    candidate_payload = [
        {
            "name": candidate.name,
            "source": candidate.source,
            "url": candidate.url,
            "description": candidate.description,
            "location": candidate.location,
            "sector": candidate.sector,
            "stage": candidate.stage,
            "tags": candidate.tags or [],
        }
        for candidate in candidates
    ]
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "system",
                "content": (
                    "You are building only the startup-search portion of an investment workflow. "
                    "Return strict JSON with keys startup_name, startup_list, evaluated_startups, startup_basic_info. "
                    "startup_name must be the single best first company to evaluate. "
                    "startup_list must be a ranked list. "
                    "evaluated_startups must be an empty list. "
                    "startup_basic_info must describe startup_name only. "
                    "Do not invent facts. If data is missing, use null or an empty string."
                ),
            },
            {
                "role": "user",
                "content": json.dumps(
                    {
                        "user_query": user_query,
                        "candidates": candidate_payload,
                    },
                    ensure_ascii=False,
                ),
            },
        ],
    )
    return json.loads(response.output_text)


search_state = build_search_state(user_query, filtered_candidates, client)
search_state

{'startup_name': 'Piggy Robotics',
 'startup_list': ['Piggy Robotics',
  'Proception Inc',
  'K-Scale Labs',
  'Vassar Robotics',
  'Human Archive',
  'Cortex AI',
  'One Robot',
  'Verne Robotics',
  'Origami Robotics',
  'Forge Robotics',
  'Hlabs',
  'Boost Robotics'],
 'evaluated_startups': [],
 'startup_basic_info': {'name': 'Piggy Robotics',
  'source': 'ycombinator',
  'url': 'https://www.ycombinator.com/companies/piggy-robotics',
  'description': 'Humanoid robots that do your chores for the price of an iPhone!',
  'location': '',
  'sector': 'Industrials -> Manufacturing and Robotics',
  'stage': 'Early',
  'tags': ['Artificial Intelligence',
   'Home Automation',
   'Robotics',
   'Smart Home Assistants']}}

## 6. 최종 확인

In [31]:
print("startup_name:", search_state["startup_name"])
print("startup_list:", search_state["startup_list"])
print("evaluated_startups:", search_state["evaluated_startups"])
print("startup_basic_info:")
print(json.dumps(search_state["startup_basic_info"], ensure_ascii=False, indent=2))

startup_name: Piggy Robotics
startup_list: ['Piggy Robotics', 'Proception Inc', 'K-Scale Labs', 'Vassar Robotics', 'Human Archive', 'Cortex AI', 'One Robot', 'Verne Robotics', 'Origami Robotics', 'Forge Robotics', 'Hlabs', 'Boost Robotics']
evaluated_startups: []
startup_basic_info:
{
  "name": "Piggy Robotics",
  "source": "ycombinator",
  "url": "https://www.ycombinator.com/companies/piggy-robotics",
  "description": "Humanoid robots that do your chores for the price of an iPhone!",
  "location": "",
  "sector": "Industrials -> Manufacturing and Robotics",
  "stage": "Early",
  "tags": [
    "Artificial Intelligence",
    "Home Automation",
    "Robotics",
    "Smart Home Assistants"
  ]
}
